Cenário completo (narrativa para vídeo e texto):
* Planta de manufatura com 5.000 sensores embarcados em motores e rolamentos.
* Objetivo: prever falhas com 48h de antecedência e reduzir downtime em 30%.

Arquitetura proposta (detalhada):
* Edge: agentes coletam e pre-filtram dados, calculam agregações básicas (mean, std) a cada 30s.
* Transporte: Kafka com retenção de 7 dias para permitir reprocessamento.
* Data Lake: Armazenamento em formato Parquet/CSV (ex: S3 público ou GitHub Raw).
* Batch/Stream: Spark para recomputação diária; Spark Structured Streaming para detecção de anomalias em tempo real.
* Model training: pipelines em Databricks/EMR; experimento versionado com MLflow.
* Serving: Seldon em Kubernetes para responder predições em baixa latência.

Exemplo de pipeline (pseudocódigo simplificado para Airflow):
* task: ingest_to_datalake
* task: validate_schema
* task: compute_features
* task: train_model_if_needed
* task: deploy_model

Código exemplo (simplificado) para leitura de dados públicos e treino rápido de modelo de baseline:

In [ ]:
# Se a biblioteca não estiver instalada
!pip install dask

Utilizaremos um dataset público de manutenção preditiva (AI4I 2020) hospedado no GitHub.

In [2]:
import pandas as pd
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

try:
    # Carrega o dataset 'penguins' embutido na biblioteca seaborn
    # dropna() remove linhas com valores ausentes para evitar erros no treino
    df = sns.load_dataset('penguins').dropna()

    # Preparamos o alvo (target): convertemos a coluna de texto 'species'
    # em códigos numéricos (0, 1, 2) que o modelo consegue entender
    df['species_enc'] = df['species'].astype('category').cat.codes

    # Selecionamos as colunas numéricas que servirão como características (features)
    feature_cols = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
    X = df[feature_cols]
    y = df['species_enc']

    # Dividimos os dados: 80% para treinar o modelo e 20% para testar sua precisão
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Inicializamos o classificador Random Forest com 100 árvores de decisão
    model = RandomForestClassifier(n_estimators=100)

    # Treinamos o modelo usando o conjunto de treino
    model.fit(X_train, y_train)

    # Calculamos a acurácia comparando as previsões com os resultados reais do teste
    accuracy = model.score(X_test, y_test)
    print(f'Modelo treinado com sucesso! Acurácia: {accuracy:.2f}')

    # Exibimos as primeiras linhas para conferir a estrutura final dos dados
    display(df.head())
except Exception as e:
    print(f'Erro ao processar o pipeline de ML: {e}')

Modelo treinado com sucesso! Acurácia: 1.00


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex,species_enc
0,Adelie,Torgersen,39.1,18.7,181.0,3750.0,Male,0
1,Adelie,Torgersen,39.5,17.4,186.0,3800.0,Female,0
2,Adelie,Torgersen,40.3,18.0,195.0,3250.0,Female,0
4,Adelie,Torgersen,36.7,19.3,193.0,3450.0,Female,0
5,Adelie,Torgersen,39.3,20.6,190.0,3650.0,Male,0


Discussão sobre balanceamento (falhas raras) e técnicas: oversampling, SMOTE, simulação de condições de falha.
Exemplo em Python – Processamento com PySpark

In [3]:
from pyspark.sql import SparkSession
import pandas as pd
import numpy as np

# Gerando um dataset bem maior (1 milhão de linhas) para simular Big Data
print("Gerando dados sintéticos...")
n_rows = 1_000_000
data = {
    'sensor_id': np.random.randint(1, 5001, n_rows),
    'temperatura': np.random.normal(75, 10, n_rows),
    'vibracao': np.random.uniform(0.1, 5.0, n_rows)
}
pd.DataFrame(data).to_csv("dados_sensores_big.csv", index=False)

# Inicializando a sessão Spark
spark = SparkSession.builder.appName("BigDataIA").getOrCreate()

# Lendo o arquivo maior
print("Lendo e processando com Spark...")
df = spark.read.csv("dados_sensores_big.csv", header=True, inferSchema=True)

# Registro da tabela temporária para SQL
df.createOrReplaceTempView("dados_industriais")

# Consulta analítica de alta performance
resultado = spark.sql("""
    SELECT
        COUNT(*) as total_registros,
        AVG(temperatura) as media_temp,
        MAX(vibracao) as max_vibracao
    FROM dados_industriais
""")

resultado.show()
print(f"Processamento de {n_rows} linhas concluído.")

Gerando dados sintéticos...
Lendo e processando com Spark...
+---------------+-----------------+-----------------+
|total_registros|       media_temp|     max_vibracao|
+---------------+-----------------+-----------------+
|        1000000|75.00804488328852|4.999987031626131|
+---------------+-----------------+-----------------+

Processamento de 1000000 linhas concluído.


Com o PySpark, é possível processar milhões de registros distribuídos entre diversos nós de computação. Esse tipo de arquitetura é essencial para análises preditivas em tempo real e para alimentar sistemas de IA industriais.